In [6]:
import numpy as np

# Fixed seed -> reproducible output every run (same result each time
# this notebook is executed, useful for grading/demo purposes)
rng = np.random.default_rng(seed=42)

## 1. Tokenize and embed the sentence

In [ ]:
sentence = "The cat sat on the mat because it was tired now the owner is looking for it"
tokens = sentence.split()
n_tokens = len(tokens)

print(f"Tokens ({n_tokens}): {tokens}")

Tokens (10): ['The', 'cat', 'sat', 'on', 'the', 'mat', 'because', 'it', 'was', 'tired']


## 2. Project into Query, Key, Value spaces

In [8]:
d_model = 8  # small embedding dim so printed numbers stay readable
             # (real transformers use 512+, per Vaswani et al. 2017)

# Random per-token embedding. In a trained model this table is learned
# (e.g. via Word2Vec-style pretraining or jointly with the transformer);
# here it's a stand-in so we can focus on the attention math itself.
X = rng.normal(size=(n_tokens, d_model))

d_k = d_model  # single-head attention, so d_k == d_model
               # (multi-head would split d_model across heads instead)

# W_Q, W_K, W_V are the learned projection matrices in a real transformer.
# Random + fixed here for the same reason as the embeddings above:
# no training loop, just illustrating the mechanism.
W_Q = rng.normal(size=(d_model, d_k)) * 0.5
W_K = rng.normal(size=(d_model, d_k)) * 0.5
W_V = rng.normal(size=(d_model, d_k)) * 0.5

Q = X @ W_Q  # what each token is "looking for"
K = X @ W_K  # what each token "offers" to be matched against
V = X @ W_V  # content that gets mixed together by the final weights

print("Q, K, V shapes:", Q.shape, K.shape, V.shape)

Q, K, V shapes: (10, 8) (10, 8) (10, 8)


## 3. Scaled dot-product attention

In [12]:
def softmax(x, axis=-1):
    # Subtract row max before exp() for numerical stability
    # (standard trick, doesn't change the result)
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Dividing by sqrt(d_k) is the "scaled" part of scaled dot-product
# attention: without it, larger d_k pushes dot products to extremes
# and softmax saturates (near one-hot, vanishing gradients in real training).
scores = (Q @ K.T) / np.sqrt(d_k)

# One attention-weight row per query token; each row sums to 1
attention_weights = softmax(scores, axis=-1)

print("Attention weight matrix shape:", attention_weights.shape)
print("Row sums (should all be 1.0):", np.round(attention_weights.sum(axis=1), 4))

Attention weight matrix shape: (10, 10)
Row sums (should all be 1.0): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 4. Text-based bar chart of attention weights

In [11]:
def print_attention_bars(query_idx, tokens, attention_weights, bar_width=40):
    # Bars are scaled relative to the row's max weight (not an absolute
    # 0-1 scale), so the dominant token's bar always fills the full width -
    # keeps the pattern readable even when attention is spread thin.
    query_token = tokens[query_idx]
    weights = attention_weights[query_idx]
    max_w = weights.max()

    print(f'Attention from query token: "{query_token}" (position {query_idx})')
    print("-" * 60)
    for tok, w in zip(tokens, weights):
        bar_len = int(round((w / max_w) * bar_width))
        bar = "#" * bar_len
        print(f"{tok:>10} | {bar} {w:.3f}")
    print()

## 5. Demo: run the visualizer on two query tokens

In [13]:
# "it" is a good example since coreference-style attention
# (pronoun -> the noun it refers to) is a classic pattern real heads learn.
for query_word in ["it", "sat"]:
    idx = tokens.index(query_word)
    print_attention_bars(idx, tokens, attention_weights)

Attention from query token: "it" (position 7)
------------------------------------------------------------
       The | ######## 0.070
       cat | # 0.005
       sat | ##### 0.046
        on | ##### 0.040
       the | ######################################## 0.358
       mat | ############ 0.107
   because | ## 0.014
        it | ################################## 0.308
       was | # 0.006
     tired | ##### 0.046

Attention from query token: "sat" (position 2)
------------------------------------------------------------
       The | ######################################## 0.184
       cat | ############################### 0.144
       sat | ############################ 0.128
        on | ##################### 0.095
       the | ############# 0.061
       mat | ############ 0.054
   because | ############################## 0.137
        it | ######## 0.036
       was | ##################### 0.095
     tired | ############## 0.066

